In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL3dlbnMvRGVza3RvcC8yMDI2L2NvbHVtYmlhX2ZhbGxfMjAyNi9wcm9qZWN0cy9TVEFUNTI5My1maW5hbC1wcm9qZWN0L25vdGVib29rcw=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/importlib/_bootstrap.py": 1787439448.9532597, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/importlib/_bootstrap_external.py": 1787439448.9536366, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/zipimport.py": 1787439449.1178052, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/codecs.py": 1787439448.880945, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/encodings/aliases.py": 1787439448.8985407, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/encodings/__init__.py": 1787439448.8977082, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/encodings/utf_8.py": 1787439448.916199, "/Users/wens/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/abc.py": 17874394

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "Python"))
from market_data.universe import get_universe

plt.rcParams.update({"figure.figsize": (6.5, 2.8), "axes.grid": True, "grid.color": "0.9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1})

returns = pd.read_parquet(ROOT / "data" / "returns" / "1h_bar.parquet")
returns["date"] = returns.index.tz_localize(None).normalize()
returns["hour"] = returns.index.hour


def table(df: pd.DataFrame, digits: int = 4) -> Markdown:
    return Markdown(df.round(digits).to_markdown())

In [3]:
keys = returns.reset_index()[["ts", "ticker"]]
bars_per_day = returns.groupby(["date", "ticker"]).size()
summary = pd.Series({
    "rows": f"{len(returns):,}",
    "tickers": returns["ticker"].nunique(),
    "trading days": returns["date"].nunique(),
    "first bar": returns.index.min(),
    "last bar": returns.index.max(),
    "NaN returns": int(returns["ret"].isna().sum()),
    "duplicate (ts, ticker)": int(keys.duplicated().sum()),
    "sorted by ts": returns.index.is_monotonic_increasing,
    "hours present": sorted(returns["hour"].unique()),
    "ticker-days with 7 bars": f"{(bars_per_day == 7).mean():.2%}",
}, name="value")
table(summary.to_frame())

|                         | value                                                                                              |
|:------------------------|:---------------------------------------------------------------------------------------------------|
| rows                    | 7,238,510                                                                                          |
| tickers                 | 664                                                                                                |
| trading days            | 2145                                                                                               |
| first bar               | 2018-05-01 11:00:00-04:00                                                                          |
| last bar                | 2026-11-10 16:00:00-05:00                                                                          |
| NaN returns             | 0                                                                                                  |
| duplicate (ts, ticker)  | 0                                                                                                  |
| sorted by ts            | True                                                                                               |
| hours present           | [np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16)] |
| ticker-days with 7 bars | 99.24%                                                                                             |

In [4]:
#| fig-cap: "Number of tickers with returns per day vs. the point-in-time universe size."
dates = returns["date"].unique()
universe_size = pd.Series({d: len(get_universe(d)) for d in dates})
with_returns = returns.groupby("date")["ticker"].nunique()
fig, ax = plt.subplots()
ax.plot(universe_size.index, universe_size, label="universe members", color="0.6")
ax.plot(with_returns.index, with_returns, label="tickers with returns", color="tab:blue")
ax.set_ylabel("tickers")
ax.legend(frameon=False)
plt.show()
shortfall = (universe_size - with_returns).describe()[["mean", "50%", "max"]]
table(shortfall.rename("members without returns").to_frame(), 2)

<Figure size 1950x840 with 1 Axes>

|      |   members without returns |
|:-----|--------------------------:|
| mean |                       0.1 |
| 50%  |                       0   |
| max  |                       2   |

In [5]:
q = returns["ret"].quantile([0.0001, 0.001, 0.01, 0.5, 0.99, 0.999, 0.9999])
q.index = [f"{p:.2%}" for p in q.index]
table(q.rename("quantile of hourly return").to_frame())

|        |   quantile of hourly return |
|:-------|----------------------------:|
| 0.01%  |                     -0.1239 |
| 0.10%  |                     -0.0564 |
| 1.00%  |                     -0.0229 |
| 50.00% |                      0.0001 |
| 99.00% |                      0.0233 |
| 99.90% |                      0.0592 |
| 99.99% |                      0.126  |

In [6]:
#| fig-cap: "Standard deviation of hourly returns by bar (10:00 includes the overnight gap)."
by_hour = returns.groupby("hour")["ret"].std()
fig, ax = plt.subplots()
ax.bar([f"{h}:00" for h in by_hour.index], by_hour * 1e4, width=0.6, color="tab:blue")
ax.set_ylabel("std (bp)")
plt.show()

<Figure size 1950x840 with 1 Axes>

In [7]:
top = returns.reset_index().assign(abs_ret=lambda d: d["ret"].abs()).nlargest(15, "abs_ret")
table(top[["ts", "ticker", "ret"]].assign(ts=lambda d: d["ts"].dt.strftime("%Y-%m-%d %H:%M")).set_index("ts"))

| ts               | ticker   |     ret |
|:-----------------|:---------|--------:|
| 2026-08-19 10:00 | MRNA     |  1.1047 |
| 2020-03-02 10:00 | IR       | -0.7511 |
| 2020-03-09 10:00 | FANG     | -0.5187 |
| 2021-06-07 14:00 | BIIB     |  0.4408 |
| 2020-03-09 10:00 | APA      | -0.4319 |
| 2020-03-09 10:00 | OXY      | -0.4311 |
| 2020-03-09 10:00 | MRO      | -0.4095 |
| 2020-03-09 10:00 | HES      | -0.4057 |
| 2023-03-13 10:00 | CMA      | -0.4032 |
| 2020-03-09 10:00 | SLB      | -0.3973 |
| 2020-06-26 10:00 | GPS      |  0.3972 |
| 2025-09-10 10:00 | ORCL     |  0.3952 |
| 2024-07-26 10:00 | DXCM     | -0.3929 |
| 2020-03-09 10:00 | HAL      | -0.3877 |
| 2020-11-04 11:00 | BIIB     |  0.3874 |

In [8]:
splits = [("AAPL", "2020-08-31", "4:1"), ("NVDA", "2021-07-20", "4:1"), ("AMZN", "2022-06-06", "20:1"),
          ("GOOGL", "2022-07-18", "20:1"), ("TSLA", "2022-08-25", "3:1"), ("WMT", "2024-02-26", "3:1"),
          ("NVDA", "2024-06-10", "10:1"), ("CMG", "2024-06-26", "50:1"), ("AVGO", "2024-07-15", "10:1")]
rows = []
for ticker, day, ratio in splits:
    r = returns[(returns["ticker"] == ticker) & (returns["date"] == pd.Timestamp(day))]
    rows.append({"ticker": ticker, "split day": day, "ratio": ratio,
                 "10:00 return": r.loc[r["hour"] == 10, "ret"].iloc[0], "max |hourly return|": r["ret"].abs().max()})
table(pd.DataFrame(rows).set_index("ticker"))

| ticker   | split day   | ratio   |   10:00 return |   max |hourly return| |
|:---------|:------------|:--------|---------------:|----------------------:|
| AAPL     | 2020-08-31  | 4:1     |         0.0267 |                0.0267 |
| NVDA     | 2021-07-20  | 4:1     |        -0.0171 |                0.0171 |
| AMZN     | 2022-06-06  | 20:1    |         0.0364 |                0.0364 |
| GOOGL    | 2022-07-18  | 20:1    |         0.0125 |                0.0211 |
| TSLA     | 2022-08-25  | 3:1     |        -0.0012 |                0.0123 |
| WMT      | 2024-02-26  | 3:1     |        -0.0003 |                0.0109 |
| NVDA     | 2024-06-10  | 10:1    |        -0.0217 |                0.0297 |
| CMG      | 2024-06-26  | 50:1    |        -0.0191 |                0.0191 |
| AVGO     | 2024-07-15  | 10:1    |        -0.0005 |                0.0118 |

In [9]:
#| fig-cap: Growth of $1 from compounded hourly returns (log scale). Split days show no jumps.
TICKERS = ["AAPL", "NVDA", "AMZN", "TSLA", "XOM", "BRK.B", "GE", "IPG"]
fig, axes = plt.subplots(4, 2, figsize=(6.5, 7), sharex=True)
for ax, ticker in zip(axes.flat, TICKERS):
    r = returns.loc[returns["ticker"] == ticker, "ret"]
    ax.plot(r.index, (1 + r).cumprod(), color="tab:blue")
    ax.set_yscale("log")
    ax.set_title(ticker, fontsize=9, loc="left")
    ax.tick_params(labelsize=7)
fig.supylabel("growth of $1", fontsize=9)
fig.tight_layout()
plt.show()

<Figure size 1950x2100 with 8 Axes>

In [10]:
raw = pd.concat(pd.read_parquet(f).query("ticker in @TICKERS")
                for f in sorted((ROOT / "data" / "XNAS.ITCH").glob("20*/all.parquet")))
rows = []
for ticker in TICKERS:
    px = raw.loc[raw["ticker"] == ticker, "close"] * raw.loc[raw["ticker"] == ticker, "adjust_factor"]
    gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap
    recomputed = (px / px.shift(1) - 1).where(~gap).dropna()
    ours = returns.loc[returns["ticker"] == ticker, "ret"]
    rows.append({"ticker": ticker, "returns in file": len(ours), "recomputed": len(recomputed),
                 "same timestamps": ours.index.equals(recomputed.index),
                 "max |difference|": f"{(ours - recomputed).abs().max():.1e}"})
table(pd.DataFrame(rows).set_index("ticker"))

/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  raw = pd.concat(pd.read_parquet(f).query("ticker in @TICKERS")


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw = pd.concat(pd.read_parquet(f).query("ticker in @TICKERS")
/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


/var/folders/5q/5sxftbp97v570grsxg72c3zr0000gn/T/ipykernel_97513/921198630.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = px.index.to_series().diff() > pd.Timedelta(days=5)  # same rule as return.py: no return across a membership gap


| ticker   |   returns in file |   recomputed | same timestamps   |   max |difference| |
|:---------|------------------:|-------------:|:------------------|-------------------:|
| AAPL     |             15014 |        15014 | True              |                  0 |
| NVDA     |             15013 |        15013 | True              |                  0 |
| AMZN     |             15013 |        15013 | True              |                  0 |
| TSLA     |             10352 |        10352 | True              |                  0 |
| XOM      |             15006 |        15006 | True              |                  0 |
| BRK.B    |             15009 |        15009 | True              |                  0 |
| GE       |             15004 |        15004 | True              |                  0 |
| IPG      |             11563 |        11563 | True              |                  0 |

In [11]:
import wrds

db = wrds.Connection(wrds_username="ws2840")
crsp = db.raw_sql("""
    select s.tradingsymbol, d.dlycaldt, d.dlyret, d.dlyorddivamt, d.dlyfacprc
    from crsp_q_stock.dsf_v2 d
    join crsp_q_stock.stksecurityinfohist s
      on d.permno = s.permno and d.dlycaldt between s.secinfostartdt and s.secinfoenddt
    where s.tradingsymbol in %(symbols)s and d.dlycaldt > '2018-05-01'
""", params={"symbols": tuple(t.replace(".", "") for t in TICKERS)}, date_cols=["dlycaldt"])
db.close()

compared, rows = {}, []
for ticker in TICKERS:
    ours = returns.loc[returns["ticker"] == ticker, "ret"]
    daily = (1 + ours).groupby(ours.index.tz_localize(None).normalize()).prod() - 1
    c = crsp[crsp["tradingsymbol"] == ticker.replace(".", "")].set_index("dlycaldt")
    both = pd.concat([daily.rename("ours"), c], axis=1).dropna(subset=["ours", "dlyret"])
    diff = (both["ours"] - both["dlyret"]).abs() * 1e4
    events = (both["dlyorddivamt"] > 0) | (both["dlyfacprc"] != 1)
    compared[ticker] = both
    rows.append({"ticker": ticker, "days": len(both), "corr": round(both["ours"].corr(both["dlyret"]), 4),
                 "median bp": round(diff.median(), 1), "95th bp": round(diff.quantile(0.95), 1),
                 "max bp": round(diff.max(), 1), "div/split days": int(events.sum()),
                 "max bp on those": round(diff[events].max(), 1) if events.any() else None})  # BRK.B: none
table(pd.DataFrame(rows).set_index("ticker"))

Loading library list...


Done


| ticker   |   days |   corr |   median bp |   95th bp |   max bp |   div/split days |   max bp on those |
|:---------|-------:|-------:|------------:|----------:|---------:|-----------------:|------------------:|
| AAPL     |   2051 | 0.9991 |         2.1 |      12.7 |    120   |               34 |               9.8 |
| NVDA     |   2051 | 0.9997 |         2.4 |      14   |     69.3 |               35 |              20.4 |
| AMZN     |   2051 | 0.9982 |         2.4 |      11.7 |    360   |                1 |               4.9 |
| TSLA     |   1386 | 0.9992 |         2.4 |       9.6 |    496.3 |                1 |               1.1 |
| XOM      |   2051 | 0.9995 |         2.2 |       9.1 |     97.9 |               33 |               5.6 |
| BRK.B    |   2051 | 0.9987 |         2.9 |      12.6 |     60.5 |                0 |             nan   |
| GE       |   2051 | 0.9984 |         3.4 |      17.4 |    371.8 |               35 |              27.6 |
| IPG      |   1656 | 0.9996 |         3   |       9.3 |     81.3 |               26 |              14   |

In [12]:
#| fig-cap: 'Daily return: compounded hourly returns (y) vs. CRSP (x), in %.'
fig, axes = plt.subplots(4, 2, figsize=(6.5, 7))
for ax, ticker in zip(axes.flat, TICKERS):
    both = compared[ticker] * 100
    ax.scatter(both["dlyret"], both["ours"], s=3, color="tab:blue", alpha=0.5)
    lim = [both[["ours", "dlyret"]].min().min(), both[["ours", "dlyret"]].max().max()]
    ax.plot(lim, lim, color="0.5", linewidth=0.8)
    ax.set_title(ticker, fontsize=9, loc="left")
    ax.tick_params(labelsize=7)
fig.supxlabel("CRSP daily return (%)", fontsize=9)
fig.supylabel("ours (%)", fontsize=9)
fig.tight_layout()
plt.show()

<Figure size 1950x2100 with 8 Axes>